In [ ]:
import pennylane as qml
from math import pi
import numpy as np

n_wires = 3
dev = qml.device('default.qubit', wires=n_wires)

@qml.qnode(dev)
def circuit(weights:np.array,reps:int=1):
    for i in range(n_wires):
        qml.Hadamard(wires=i)
    
    for i in range(n_wires):
        qml.RX(weights[0][i], wires=i)
        qml.RZ(weights[1][i], wires=i)
    
    for i in range(n_wires-1):
        qml.CNOT(wires=[i, i+1])
        
    for i in range(n_wires):
        qml.RY(weights[2][i], wires=i)
        qml.RX(weights[3][i], wires=i)

    return 



In [ ]:
weights = np.random.random(size=(4,3))
print(qml.draw(circuit)(np.random.random(size=(4,3))))

In [ ]:
import matplotlib.pyplot as plt
qml.drawer.use_style("black_white")
fig, ax = qml.draw_mpl(circuit)(weights)
plt.show()

In [ ]:
decomposed_circuit = qml.transforms.decompose(circuit) 

In [ ]:
print(qml.draw(decomposed_circuit)([[pi, pi, pi]]))

In [11]:
import pennylane as qml
from pennylane import numpy as np

def hardware_efficient_ansatz(params, reps, rotation_gates, entanglement_strategy, wires):
    for rep in range(reps):
        # 应用旋转门
        for wire in wires:
            for i, gate in enumerate(rotation_gates):
                if gate == 'RX':
                    qml.RX(params[rep, i, wire], wires=wire)
                elif gate == 'RY':
                    qml.RY(params[rep, i, wire], wires=wire)
                elif gate == 'RZ':
                    qml.RZ(params[rep, i, wire], wires=wire)
                else:
                    raise ValueError(f"Unsupported rotation gate: {gate}")

        # 应用纠缠门
        if entanglement_strategy == 'full':
            for i in range(len(wires)):
                for j in range(i + 1, len(wires)):
                    qml.CNOT(wires=[wires[i], wires[j]])
        elif entanglement_strategy == 'linear':
            for i in range(len(wires) - 1):
                qml.CNOT(wires=[wires[i], wires[i + 1]])
        else:
            raise ValueError(f"Unsupported entanglement strategy: {entanglement_strategy}")

# 定义量子设备
dev = qml.device("default.qubit", wires=4)

# 创建 QNode
@qml.qnode(dev)
def circuit(params, reps, rotation_gates, entanglement_strategy, c):
    hardware_efficient_ansatz(params, reps, rotation_gates, entanglement_strategy, wires=[0, 1, 2, 3])
    # 计算每个 Z 测量的期望值并乘以相应的系数 ci
    expvals = [qml.expval(qml.PauliZ(i)) for i in range(4)]
    # 加和加权的结果
    return qml.math.dot(c, expvals)

# 用户提供的参数
reps = 2
rotation_gates = ['RX', 'RY']
entanglement_strategy = 'linear'
params = np.random.normal(0, np.pi, (reps, len(rotation_gates), 4))

# 系数 ci
c = np.array([0.5, 0.5, 0.5, 0.5])  # 示例系数，可以根据需要调整

# 测试电路
print(qml.draw(circuit)(params, reps, rotation_gates, entanglement_strategy, c))
print(circuit(params, reps, rotation_gates, entanglement_strategy, c))


TypeError: unsupported operand type(s) for *: 'float' and 'ExpectationMP'

In [13]:
import pennylane as qml
from pennylane import numpy as np
import jax
import jax.numpy as jnp
import optax

# 设置随机种子
key = jax.random.PRNGKey(42)

# 定义量子设备
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

# 定义参数化量子电路（Hardware-Efficient Ansatz）
def hardware_efficient_ansatz(params, reps, rotation_gates, entanglement_strategy, wires):
    for rep in range(reps):
        # 应用旋转门
        for wire in wires:
            for i, gate in enumerate(rotation_gates):
                if gate == 'RX':
                    qml.RX(params[rep, i, wire], wires=wire)
                elif gate == 'RY':
                    qml.RY(params[rep, i, wire], wires=wire)
                elif gate == 'RZ':
                    qml.RZ(params[rep, i, wire], wires=wire)
        # 应用纠缠门
        if entanglement_strategy == 'full':
            for i in range(len(wires)):
                for j in range(i + 1, len(wires)):
                    qml.CNOT(wires=[wires[i], wires[j]])
        elif entanglement_strategy == 'linear':
            for i in range(len(wires) - 1):
                qml.CNOT(wires=[wires[i], wires[i + 1]])

# 定义量子神经网络（QNN）
@qml.qnode(dev, interface="jax")
def quantum_neural_network(params, reps, rotation_gates, entanglement_strategy, c):
    wires = range(n_qubits)
    hardware_efficient_ansatz(params, reps, rotation_gates, entanglement_strategy, wires)
    # 测量每个 Z_i 并乘以 c_i
    expvals = [qml.expval(qml.PauliZ(i)) for i in wires]
    # 使用 jax.numpy 进行加权和
    return jnp.dot(jnp.array(c), jnp.array(expvals))

# 初始化参数
reps = 2
rotation_gates = ['RX', 'RY']
entanglement_strategy = 'full'
params = jax.random.normal(key, (reps, len(rotation_gates), n_qubits))
c = jnp.array([0.5, 0.5, 0.5, 0.5])  # 可训练系数

# 定义损失函数（示例：最小化输出）
def loss_fn(params, c):
    return quantum_neural_network(params, reps, rotation_gates, entanglement_strategy, c)

# 定义优化器
optimizer = optax.adam(learning_rate=0.01)
opt_state = optimizer.init((params, c))

# 训练循环
@jax.jit
def update_step(params, c, opt_state):
    loss, grads = jax.value_and_grad(loss_fn, argnums=(0, 1))((params, c))
    updates, opt_state = optimizer.update(grads, opt_state)
    params, c = optax.apply_updates((params, c), updates)
    return params, c, opt_state, loss



In [14]:
print(qml.draw(circuit)(params, reps, rotation_gates, entanglement_strategy, c))

TypeError: dot requires ndarray or scalar arguments, got <class 'list'> at position 1.